# Week 3 — Geospatial join (stations + demand 2024)

**Objective:** join BA Subway stations (geolocated) with 2024 demand to produce an interactive map and exportables.

**Inputs (raw):**
- `data/raw/estaciones/bocas-de-subte.csv` (station entrances with lat/lon)
- `data/raw/molinetes/*.csv` (turnstile counts, 2024)

**Outputs (processed):**
- `data/processed/map_stations_demand_2024.png` (static chart for README/LinkedIn)
- `data/processed/stations_with_demand_2024.csv` (joined dataset)
- `assets/screenshots/week3_map.png` (screenshot for README)

**Notes:** deterministic normalization via global + per-(line,station) aliases. If GCBA changes names next year, update alias dictionaries.


In [1]:
import os, re, unicodedata, glob
import pandas as pd
import plotly.express as px


In [2]:
BASE_DIR = os.path.abspath("..")
RAW_DIR  = os.path.join(BASE_DIR, "data", "raw")
PROC_DIR = os.path.join(BASE_DIR, "data", "processed")
ASSETS_DIR = os.path.join(BASE_DIR, "assets", "screenshots")

MOL_DIR   = os.path.join(RAW_DIR, "molinetes")
BOCAS_CSV = os.path.join(RAW_DIR, "estaciones", "bocas-de-subte.csv")

os.makedirs(PROC_DIR, exist_ok=True)
os.makedirs(ASSETS_DIR, exist_ok=True)


In [3]:
def strip_accents(s: str) -> str:
    if not isinstance(s, str): return s
    return ''.join(ch for ch in unicodedata.normalize('NFD', s) if unicodedata.category(ch) != 'Mn')

def norm_key(s: str) -> str:
    s = (s or "").upper().strip()
    s = strip_accents(s)
    s = s.replace("°","")
    s = re.sub(r"\s+", " ", s)
    return s


In [ ]:
import re

print("Columns in bocas:", list(bocas.columns))

# normalizamos a minúsculas para detectar
bocas_cols_lower = {c.lower(): c for c in bocas.columns}

def pick_col(candidates):
    """Devuelve el nombre REAL de la primera columna que matchee alguno de los candidatos (en minúscula)."""
    for cand in candidates:
        if cand in bocas_cols_lower:
            return bocas_cols_lower[cand]
    # si no match exacto, probamos 'contiene'
    for c_lower, c_real in bocas_cols_lower.items():
        if any(cand in c_lower for cand in candidates):
            return c_real
    return None

# candidatos más amplios
lat_col = pick_col(["lat","latitude","latitud","y"])
lon_col = pick_col(["lon","long","longitude","longitud","x"])
sta_col = pick_col(["estacion","estación","nombre_estacion","nom_estacion","station","nombre"])
lin_col = pick_col(["linea","línea","line","linea_desc","linea_nombre"])

# Si no hay lat/lon, intentamos parsear geometry tipo POINT(lon lat)
geom_col = pick_col(["geometry","geom","wkt","coordenadas","coord","point"])

# Clonamos para no tocar el DF original
b = bocas.copy()

# Si faltan lat/lon pero hay geometry en WKT, parseamos
if (lat_col is None or lon_col is None) and geom_col is not None:
    # intento extraer con regex POINT(lon lat)
    def parse_point(val):
        if not isinstance(val, str):
            return None, None
        m = re.search(r"POINT\s*\(\s*([-\d\.]+)\s+([-\d\.]+)\s*\)", val, re.IGNORECASE)
        if m:
            lon = float(m.group(1))
            lat = float(m.group(2))
            return lat, lon
        return None, None
    lat_vals, lon_vals = [], []
    for v in b[geom_col]:
        latv, lonv = parse_point(v)
        lat_vals.append(latv)
        lon_vals.append(lonv)
    b["__lat__geom"] = lat_vals
    b["__lon__geom"] = lon_vals
    if lat_col is None:
        lat_col = "__lat__geom"
    if lon_col is None:
        lon_col = "__lon__geom"

# Validaciones y mensajes útiles
if sta_col is None:
    raise KeyError("No pude detectar la columna de estación (ej: 'estacion', 'estación', 'station'). Revisa nombres.")
if lin_col is None:
    raise KeyError("No pude detectar la columna de línea (ej: 'linea', 'línea', 'line'). Revisa nombres.")
if lat_col is None or lon_col is None:
    raise KeyError("No pude detectar columnas de coordenadas (lat/lon), ni parsear geometry. Revisa nombres.")

print("Detected columns →",
      "station:", sta_col, "| line:", lin_col, "| lat:", lat_col, "| lon:", lon_col)

# Renombramos a canónico y tomamos solo lo necesario
bocas_geo = b.rename(columns={
    sta_col: "station",
    lin_col: "line",
    lat_col: "lat",
    lon_col: "lon",
})[["station","line","lat","lon"]].copy()

# Normalizamos
bocas_geo["station"] = bocas_geo["station"].astype(str).str.strip().str.upper()
bocas_geo["line"]    = bocas_geo["line"].astype(str).str.upper().str.replace(r"^LINEA\s*", "", regex=True)

# Simplificado para join
bocas_geo["station_simpl"] = bocas_geo["station"].apply(norm_key)

# Un punto promedio por estación/linea
bocas_geo_agg = (
    bocas_geo.groupby(["station","line","station_simpl"], as_index=False)[["lat","lon"]]
    .mean()
)

print("bocas_geo_agg shape:", bocas_geo_agg.shape)
display(bocas_geo_agg.head())


Columns in bocas: ['long', 'lat', 'id', 'linea', 'estacion', 'numero_de_', 'destino_bo', 'lineas_de_', 'cierra_fin', 'escalera_p', 'escalera_m', 'ascensor', 'rampa', 'salvaescal', 'calle', 'altura', 'calle2', 'barrio', 'comuna', 'observacio', 'objeto', 'dom_norma', 'dom_orig']
Detected columns → station: estacion | line: linea | lat: lat | lon: long
bocas_geo_agg shape: (90, 5)


,station,line,station_simpl,lat,lon
0,9 DE JULIO,D,9 DE JULIO,-34.604382,-58.380540
1,ACOYTE,A,ACOYTE,-34.618065,-58.435926
2,AGÜERO,D,AGUERO,-34.591699,-58.407095
3,ALBERTI,A,ALBERTI,-34.609927,-58.400884
4,ALMAGRO - MEDRANO,B,ALMAGRO - MEDRANO,-34.603214,-58.420972


In [ ]:
# === Cargar demanda 2024 ===
import os, glob, re, csv
import pandas as pd

RAW_DIR   = os.path.join(BASE_DIR, "data", "raw")
MOL_DIR   = os.path.join(RAW_DIR, "molinetes")
PROC_DIR  = os.path.join(BASE_DIR, "data", "processed")
os.makedirs(PROC_DIR, exist_ok=True)

# --- utilidades de columnas
from collections import defaultdict

def normalize_token(s: str) -> str:
    s = (s or "").strip().strip('"').strip()
    s = re.sub(r"\s+", "_", s)
    s = s.lower()
    return s or "col"

def make_unique(cols):
    seen = defaultdict(int)
    out = []
    for c in cols:
        base = normalize_token(c)
        seen[base] += 1
        out.append(base if seen[base] == 1 else f"{base}_{seen[base]-1}")
    return out

# --- leer primera línea (header) intentando varios encodings
def read_first_line_smart(path, encodings=("utf-8-sig","utf-8","latin1","cp1252")):
    for enc in encodings:
        try:
            with open(path, "r", encoding=enc) as f:
                return f.readline().rstrip("\n\r"), enc
        except Exception:
            continue
    # último recurso: binario → latin1 ignorando errores
    with open(path, "rb") as f:
        raw = f.readline()
    return raw.decode("latin1", errors="ignore").rstrip("\n\r"), "latin1"

def read_header_names_smart(path, sep=";"):
    first_line, enc = read_first_line_smart(path)
    if first_line.startswith('"') and first_line.endswith('"'):
        first_line = first_line[1:-1]
    raw_cols = [c for c in first_line.split(sep)]
    return make_unique(raw_cols), enc

# --- lector principal (sin low_memory y con quoting=NONE)
def read_molinetes_quoted_smart(path, sep=";"):
    cols, detected_enc = read_header_names_smart(path, sep=sep)
    encodings_try = [detected_enc, "utf-8-sig", "utf-8", "latin1", "cp1252"]
    tried = set()
    for enc in encodings_try:
        if enc in tried:
            continue
        tried.add(enc)
        try:
            df = pd.read_csv(
                path,
                sep=sep,
                encoding=enc,
                engine="python",
                header=None,
                names=cols,
                quoting=csv.QUOTE_NONE,
                on_bad_lines="skip"
            )
            # limpieza básica de strings
            for c in df.select_dtypes(include="object").columns:
                df[c] = df[c].astype(str).str.strip('"').str.strip()
            return df
        except Exception:
            continue
    # último recurso: leer binario → decodificar latin1 ignorando errores y parsear con pandas
    with open(path, "rb") as f:
        text = f.read().decode("latin1", errors="ignore")
    # guardar temporal y leer
    tmp_path = os.path.join(PROC_DIR, "_tmp_decode_latin1.csv")
    with open(tmp_path, "w", encoding="latin1") as f:
        f.write(text)
    df = pd.read_csv(
        tmp_path,
        sep=sep,
        encoding="latin1",
        engine="python",
        header=None,
        names=cols,
        quoting=csv.QUOTE_NONE,
        on_bad_lines="skip"
    )
    for c in df.select_dtypes(include="object").columns:
        df[c] = df[c].astype(str).str.strip('"').str.strip()
    return df

# --- cargar todos los CSVs y concatenar
csvs = sorted(glob.glob(os.path.join(MOL_DIR, "*.csv")))
print("CSV detectados:", len(csvs))
assert csvs, "No hay CSVs en data/raw/molinetes"

df_list = []
ok, fail = 0, 0
for p in csvs:
    try:
        df = read_molinetes_quoted_smart(p, sep=";")
        df["source_file"] = os.path.basename(p)
        df_list.append(df)
        ok += 1
    except Exception as e:
        print("FAIL:", os.path.basename(p), "→", e)
        fail += 1

print(f"OK files: {ok} | Failed: {fail}")
assert ok > 0, "No se pudo leer ningún CSV (revisar encodings)."

mol_full = pd.concat(df_list, ignore_index=True)

# --- canonicalizar columnas y tipos (igual que antes)
mol_full.columns = [c.strip().lower() for c in mol_full.columns]
rename_map = {}
for c in mol_full.columns:
    if c in {"fecha"}: rename_map[c] = "date"
    if c in {"desde", "desde_hora", "hora_desde"}: rename_map[c] = "time_from"
    if c in {"hasta", "hasta_hora", "hora_hasta"}: rename_map[c] = "time_to"
    if c in {"linea", "línea", "line"}: rename_map[c] = "line"
    if c in {"estacion", "estación", "station"}: rename_map[c] = "station"
    if c in {"pax_total","viajes","pasajeros","pax","passengers","conteo","count"}:
        rename_map[c] = "passengers"
mol_full = mol_full.rename(columns=rename_map)

# limpiar filas-header fantasma
mask_header_row = (
    mol_full.get("time_from", "").astype(str).str.upper().eq("DESDE") |
    mol_full.get("time_to", "").astype(str).str.upper().eq("HASTA")
)
mol_full = mol_full.loc[~mask_header_row].copy()

# tipos y normalización
mol_full["date"] = pd.to_datetime(mol_full.get("date"), errors="coerce", dayfirst=True)
mol_full["year_month"] = mol_full["date"].dt.to_period("M").astype(str)

for col in ["time_from", "time_to", "station"]:
    if col in mol_full.columns:
        mol_full[col] = mol_full[col].astype(str).str.strip().str.upper()

if "passengers" not in mol_full.columns and "pax_total" in mol_full.columns:
    mol_full["passengers"] = pd.to_numeric(mol_full["pax_total"], errors="coerce")
else:
    mol_full["passengers"] = pd.to_numeric(mol_full.get("passengers"), errors="coerce")

if "line" in mol_full.columns:
    mol_full["line"] = (mol_full["line"]
                        .str.upper()
                        .str.replace(r"^LINEA\s*", "", regex=True)
                        .str.strip())

print("Shape mol_full:", mol_full.shape)
display(mol_full.head())


CSV detectados: 24
OK files: 24 | Failed: 0
Shape mol_full: (11440440, 14)


,date,time_from,time_to,line,molinete,station,pax_pagos,pax_pases_pagos,pax_franq,passengers,source_file,col,col_1,year_month
1,2024-01-01,07:45:00,08:00:00,B,LineaB_Malabia_N_Turn01,MALABIA,3,0,0,3,202401_PAX15min-ABC.csv,NaN,NaN,2024-01
2,2024-01-01,07:45:00,08:00:00,B,LineaB_Tronador_Turn01,TRONADOR,1,0,0,1,202401_PAX15min-ABC.csv,NaN,NaN,2024-01
3,2024-01-01,07:45:00,08:00:00,B,LineaB_Pellegrini_E_Turn05,CARLOS PELLEGRINI,13,0,0,13,202401_PAX15min-ABC.csv,NaN,NaN,2024-01
4,2024-01-01,07:45:00,08:00:00,A,LineaA_Flores_Este_Turn03,FLORES,2,0,0,2,202401_PAX15min-ABC.csv,NaN,NaN,2024-01
5,2024-01-01,07:45:00,08:00:00,B,LineaB_Dorrego_N_Turn03,DORREGO,1,0,0,1,202401_PAX15min-ABC.csv,NaN,NaN,2024-01


In [8]:
# Global
alias_global = {
    "SAN JOSE DE FLORES": "FLORES",
    "ALMAGRO": "MEDRANO",
    "C. PELLEGRINI": "CARLOS PELLEGRINI",
    "DE LOS INCAS": "LOS INCAS",
    "JUAN MANUEL DE ROSAS": "ROSAS",
    "AV. DE MAYO": "AVENIDA DE MAYO",
    "MORENO": "MARIANO MORENO",
    "SAN MARTIN": "GENERAL SAN MARTIN",
    "R.SCALABRINI ORTIZ": "SCALABRINI ORTIZ",
    "AV. LA PLATA": "AVENIDA LA PLATA",
    "PLAZA DE LOS VIRREYES": "PZA. DE LOS VIRREYES",
    "BELGRANO": "GENERAL BELGRANO",
    "INDEPENDENCIA": "INDEPENDENCIA",
    "RETIRO": "RETIRO",
    "HUMBERTO 1": "HUMBERTO I",
    "PARQUE PATRICIOS": "PATRICIOS",
}

bocas_g = bocas_geo_agg.copy()
bocas_g["station_simpl"] = bocas_g["station_simpl"].map(lambda s: alias_global.get(s, s))

# Por pareja
pair_alias = {
    ("B", "CALLAO"): "CALLAO.B",
    ("B", "PUEYRREDON"): "PUEYRREDON.B",
    ("C", "INDEPENDENCIA"): "INDEPENDENCIA.C",
    ("C", "RETIRO"): "RETIRO.C",
    ("D", "PUEYRREDON"): "PUEYRREDON.D",
    ("E", "INDEPENDENCIA"): "INDEPENDENCIA.E",
    ("E", "RETIRO"): "RETIRO.E",
}

bocas_gp = bocas_g.copy()
bocas_gp["station_simpl"] = bocas_gp.apply(
    lambda r: pair_alias.get((r["line"], r["station_simpl"]), r["station_simpl"]), axis=1
)


In [11]:
# -- Mini-aggregate para este notebook --
# Requiere: mol_full (del Bloque 4) y norm_key (definido en Bloque 2)

# 1) Agregado por estación + línea (todo 2024)
agg_station_total = (
    mol_full.groupby(["station", "line"], as_index=False)["passengers"]
            .sum()
)

# 2) Clave normalizada para el join
agg_station_total["station_simpl"] = agg_station_total["station"].apply(norm_key)

# 3) Dataset limpio para merge con bocas
demand_clean = agg_station_total[["station_simpl", "line", "passengers"]].copy()

print("demand_clean shape:", demand_clean.shape)
display(demand_clean.head())


demand_clean shape: (96, 3)


,station_simpl,line,passengers
0,9 DE JULIO,D,757528
1,ACOYTE,A,2929383
2,AGUERO,D,1494099
3,ALBERTI,A,1009790
4,ANGEL GALLARDO,B,2790345


In [12]:
geo_join = pd.merge(
    bocas_gp,
    demand_clean[["station_simpl","line","passengers"]],
    on=["station_simpl","line"],
    how="left"
)
geo_join["passengers"] = geo_join["passengers"].fillna(0).astype("int64")

ratio = geo_join["passengers"].notna().mean()
print(f"Match ratio: {ratio:.1%}  ({geo_join['passengers'].notna().sum()} / {len(geo_join)})")

remaining = (geo_join[geo_join["passengers"].isna()][["station","line","station_simpl"]]
             .drop_duplicates().sort_values(["line","station"]))
remaining.head(10)


Match ratio: 100.0%  (90 / 90)


,station,line,station_simpl


In [13]:
fig_map = px.scatter_mapbox(
    geo_join, lat="lat", lon="lon", color="line", size="passengers",
    hover_name="station",
    hover_data={"line": True, "passengers": True, "lat": False, "lon": False},
    zoom=10, height=700
)
fig_map.update_layout(mapbox_style="open-street-map", margin=dict(l=0,r=0,t=40,b=0),
                      title="BA Subway — Demand by Station (2024)")

out_png = os.path.join(PROC_DIR, "map_stations_demand_2024.png")
fig_map.write_image(out_png, scale=2)

geo_join[["station","line","lat","lon","passengers"]].to_csv(
    os.path.join(PROC_DIR, "stations_with_demand_2024.csv"),
    index=False, encoding="utf-8"
)

import shutil
shot = os.path.join(ASSETS_DIR, "week3_map.png")
shutil.copyfile(out_png, shot)

print("Saved:", out_png, "and", shot)


Saved: c:\Users\do_ch\OneDrive\Escritorio\Proyectos\Proyectos GitHub\subte-dashboard\data\processed\map_stations_demand_2024.png and c:\Users\do_ch\OneDrive\Escritorio\Proyectos\Proyectos GitHub\subte-dashboard\assets\screenshots\week3_map.png
